# Week 3 Lab: Data-Driven Finance — Can AI Beat the Market?

**BUS 696: Generative AI in Finance**  
**Professor Jonathan Hersh**

---

## Learning Objectives

By the end of this lab, you will be able to:
1. Test the **random walk hypothesis** using autocorrelation and regression
2. Engineer **features** from raw price data (SMA, volatility, momentum)
3. Build and compare **ML models** for market prediction (Logistic Regression, Lasso, Random Forest, XGBoost)
4. Understand **in-sample vs. out-of-sample** evaluation and see overfitting in action
5. Evaluate whether ML models can actually **beat buy-and-hold**
6. Recognize the **data snooping trap** and why multiple testing is dangerous
7. Implement **walk-forward validation** to stress-test model performance over time
8. Use **prompt engineering** to connect empirical results to finance theory

---

## Part 1: Setup

In [ ]:
# Install if needed (run once, then comment out)
# !pip install yfinance pandas matplotlib scikit-learn xgboost statsmodels

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.tsaplots import plot_acf
import warnings
warnings.filterwarnings('ignore')

# Try importing XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
    print('XGBoost loaded successfully!')
except ImportError:
    HAS_XGBOOST = False
    print('XGBoost not installed. Run: pip install xgboost')
    print('You can still complete the lab without it.\n')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('All libraries loaded!')

In [ ]:
# Download data we'll use throughout the lab
# SPY (S&P 500 ETF) — 5 years for enough training data
spy = yf.download('SPY', period='5y')
spy.columns = spy.columns.get_level_values(0)

# Calculate returns
spy['Return'] = spy['Close'].pct_change()
spy['Log_Return'] = np.log(spy['Close'] / spy['Close'].shift(1))

print(f'SPY: {len(spy)} trading days')
print(f'From {spy.index[0].date()} to {spy.index[-1].date()}')
print(f'\nAverage daily log return: {spy["Log_Return"].mean():.5f} ({spy["Log_Return"].mean()*252:.1%} annualized)')
print(f'Daily volatility: {spy["Log_Return"].std():.4f} ({spy["Log_Return"].std()*np.sqrt(252):.1%} annualized)')
print(f'\nUp days: {(spy["Log_Return"] > 0).sum()} ({(spy["Log_Return"] > 0).mean():.1%})')
print(f'Down days: {(spy["Log_Return"] < 0).sum()} ({(spy["Log_Return"] < 0).mean():.1%})')

**Quick note on log returns:** We use log returns throughout this lab because:
- They're **additive** over time (weekly log return = sum of daily log returns)
- They're more **symmetric** (a +5% and -5% log return are equal magnitude)
- They have better **statistical properties** for modeling
- For small returns, log return ≈ simple return

The sign (positive/negative) is the same for both, so for predicting direction (up/down), it doesn't matter which we use.

---

## Part 2: Testing the Random Walk Hypothesis

The **Efficient Market Hypothesis (EMH)** in its weak form says that past prices contain no information about future returns. If this is true, returns should look like a **random walk** — each day's return is independent of previous days.

Let's test this with real data!

### 2a. Autocorrelation — Are Returns Correlated Across Days?

**Autocorrelation** measures whether today's return is correlated with yesterday's (lag 1), the day before's (lag 2), etc.

If markets are weakly efficient, all autocorrelations should be approximately **zero** (within the blue confidence bands).

In [ ]:
log_ret = spy['Log_Return'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Autocorrelation of RETURNS (direction)
plot_acf(log_ret, lags=20, ax=axes[0], title='')
axes[0].set_title('Daily Log Returns\n(Is DIRECTION predictable?)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Lag (days)')
axes[0].set_ylabel('Autocorrelation')

# Right: Autocorrelation of SQUARED returns (volatility)
plot_acf(log_ret**2, lags=20, ax=axes[1], title='')
axes[1].set_title('Squared Returns (Volatility)\n(Is MAGNITUDE predictable?)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Lag (days)')
axes[1].set_ylabel('Autocorrelation')

plt.tight_layout()
plt.show()

print('LEFT: Returns show almost no autocorrelation → you can\'t predict DIRECTION from past returns.')
print('RIGHT: Squared returns show strong autocorrelation → you CAN predict VOLATILITY!')
print('\nThis is a key insight: predicting volatility is much easier than predicting direction.')
print('(This is why the "smart money" slide said: predict something easier than returns.)')

### 2b. Can Yesterday's Return Predict Today's?

Let's be concrete: run a simple linear regression of today's return on yesterday's.

$$r_t = \alpha + \beta \cdot r_{t-1} + \epsilon$$

If $\beta \approx 0$ and $R^2 \approx 0$, then yesterday tells us nothing about today.

In [ ]:
# Create lagged return
df_lag = pd.DataFrame({
    'today': log_ret.values[1:],
    'yesterday': log_ret.values[:-1]
})

# Run regression
slope, intercept, r_value, p_value, std_err = stats.linregress(df_lag['yesterday'], df_lag['today'])

# Plot
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(df_lag['yesterday'], df_lag['today'], alpha=0.15, s=8, color='steelblue', rasterized=True)

# Regression line
x_line = np.linspace(df_lag['yesterday'].min(), df_lag['yesterday'].max(), 100)
ax.plot(x_line, intercept + slope * x_line, 'r-', linewidth=2.5, 
        label=f'OLS: β={slope:.4f}, R²={r_value**2:.5f}')

ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axvline(x=0, color='gray', linewidth=0.5)
ax.set_xlabel("Yesterday's Log Return", fontsize=13)
ax.set_ylabel("Today's Log Return", fontsize=13)
ax.set_title("Can Yesterday's Return Predict Today's?\nSPY (S&P 500 ETF)", fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f'Regression results:')
print(f'  Slope (β):  {slope:.6f}')
print(f'  R-squared:  {r_value**2:.6f}')
print(f'  p-value:    {p_value:.4f}')
print(f'\nInterpretation: Yesterday\'s return explains {r_value**2*100:.4f}% of today\'s return.')
print('That\'s essentially ZERO predictive power. The EMH appears to hold!')

### 2c. OLS with Multiple Lags

Maybe one day isn't enough. What if we use the last 5 days?

In [ ]:
from sklearn.linear_model import LinearRegression

# Create 5 lagged features
n_lags = 5
df_lags = pd.DataFrame({'target': log_ret})
for lag in range(1, n_lags + 1):
    df_lags[f'lag_{lag}'] = log_ret.shift(lag)
df_lags = df_lags.dropna()

feature_names = [f'lag_{i}' for i in range(1, n_lags + 1)]
X = df_lags[feature_names]
y = df_lags['target']

# Fit OLS
ols = LinearRegression().fit(X, y)

print('OLS Regression: Today\'s return = f(last 5 days\' returns)')
print('=' * 50)
for name, coef in zip(feature_names, ols.coef_):
    print(f'  {name}: {coef:+.6f}')
print(f'  intercept: {ols.intercept_:+.6f}')
print(f'\n  R² = {ols.score(X, y):.6f}')
print(f'\nAll coefficients are tiny. R² is essentially zero.')
print('Adding more lags doesn\'t help — past returns simply don\'t predict future returns.')
print('This is consistent with the weak form of market efficiency.')

---

## Part 3: Feature Engineering

If raw past returns can't predict the future, maybe we need **richer features** — technical indicators that capture different aspects of market behavior.

This is the standard approach in quantitative finance: transform raw data into informative features.

In [ ]:
def engineer_features(df, window=20, n_lags=5):
    """Create ML features from raw price/return data."""
    data = df[['Close', 'Log_Return']].copy().dropna()
    
    # --- Technical Indicators ---
    # SMA ratio: is price above or below its moving average?
    data['sma_ratio'] = data['Close'] / data['Close'].rolling(window).mean()
    
    # Rolling volatility: how turbulent is the market?
    data['volatility'] = data['Log_Return'].rolling(window).std()
    
    # Momentum: average of recent returns (trending up or down?)
    data['momentum'] = data['Log_Return'].rolling(window).mean()
    
    # RSI (Relative Strength Index): overbought/oversold indicator
    delta = data['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    data['rsi'] = 100 - (100 / (1 + gain / loss))
    
    # --- Create Lagged Features ---
    # We lag everything so we only use PAST information (no look-ahead bias!)
    feature_cols = []
    for col in ['Log_Return', 'sma_ratio', 'volatility', 'momentum']:
        for lag in range(1, n_lags + 1):
            name = f'{col}_lag{lag}'
            data[name] = data[col].shift(lag)
            feature_cols.append(name)
    
    # --- Target Variable ---
    # 1 if today's return is positive (market went up), 0 if negative
    data['target'] = (data['Log_Return'] > 0).astype(int)
    
    data = data.dropna()
    return data, feature_cols

# Build features
df_ml, feature_cols = engineer_features(spy)

print(f'Dataset: {len(df_ml)} samples')
print(f'Features: {len(feature_cols)}')
print(f'\nFeature list:')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')
print(f'\nTarget: up/down (1/0)')
print(f'  Up days: {df_ml["target"].mean():.1%}')
print(f'  Down days: {1 - df_ml["target"].mean():.1%}')

### Visualize the Features

Let's see what these features look like for the most recent year.

In [ ]:
recent = df_ml.iloc[-252:]  # last year

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Price + SMA
ax = axes[0]
ax.plot(recent.index, recent['Close'], color='steelblue', linewidth=1.5, label='Close Price')
sma = recent['Close'].rolling(20).mean()
ax.plot(recent.index, sma, color='red', linewidth=1.5, linestyle='--', label='20-day SMA')
ax.set_ylabel('Price ($)')
ax.set_title('SPY — Feature Engineering from Raw Price Data', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')

# Log Returns
ax = axes[1]
colors = ['green' if r > 0 else 'red' for r in recent['Log_Return']]
ax.bar(recent.index, recent['Log_Return'], color=colors, alpha=0.7, width=1)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('Log Return')
ax.set_title('Daily Log Returns (what we\'re trying to predict)', fontweight='bold')

# Volatility
ax = axes[2]
ax.fill_between(recent.index, 0, recent['volatility'], color='orange', alpha=0.5)
ax.set_ylabel('20-day Vol')
ax.set_title('Rolling Volatility — Notice the clustering! (predictable)', fontweight='bold')

# Momentum
ax = axes[3]
pos = recent['momentum'].clip(lower=0)
neg = recent['momentum'].clip(upper=0)
ax.fill_between(recent.index, 0, pos, color='green', alpha=0.5, label='Bullish')
ax.fill_between(recent.index, 0, neg, color='red', alpha=0.5, label='Bearish')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_ylabel('Momentum')
ax.set_title('20-day Momentum (Rolling Mean of Log Returns)', fontweight='bold')
ax.legend(loc='upper left')
ax.set_xlabel('Date')

plt.tight_layout()
plt.show()

---

## Part 4: The ML Model Horse Race

Now for the main event. We'll train four different ML models to predict whether SPY will go **up or down** tomorrow, using our engineered features.

**The models:**
1. **Logistic Regression** — simple linear classifier
2. **Lasso (L1 Logistic)** — automatically drops useless features
3. **Random Forest** — captures non-linear patterns with ensembles of trees
4. **XGBoost** — gradient-boosted trees, the "Kaggle champion"

**Critical**: We split into **training** (everything except the last year) and **test** (last year only). The model never sees the test data during training.

In [ ]:
# ============================================================
# TRAIN / TEST SPLIT
# ============================================================
# Use the last year as the test set (out-of-sample)
split_idx = len(df_ml) - 252
train = df_ml.iloc[:split_idx]
test = df_ml.iloc[split_idx:]

X_train = train[feature_cols]
y_train = train['target']
X_test = test[feature_cols]
y_test = test['target']

# Standardize features (important for logistic regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set: {len(train)} samples ({train.index[0].date()} to {train.index[-1].date()})')
print(f'Test set:     {len(test)} samples ({test.index[0].date()} to {test.index[-1].date()})')
print(f'Features:     {len(feature_cols)}')
print(f'\nBase rate (% of up days in test): {y_test.mean():.1%}')
print('(A model that always predicts "up" would get this accuracy.)')

In [ ]:
# ============================================================
# TRAIN ALL MODELS
# ============================================================
models = {}
results = []

# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
models['Logistic Regression'] = ('scaled', lr)

# 2. Lasso (L1-penalized logistic regression)
lasso = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, max_iter=1000, random_state=42)
lasso.fit(X_train_scaled, y_train)
models['Lasso (L1)'] = ('scaled', lasso)

# 3. Random Forest
rf = RandomForestClassifier(n_estimators=500, max_depth=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
models['Random Forest'] = ('raw', rf)

# 4. XGBoost (if available)
if HAS_XGBOOST:
    xgb = XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.05, 
                        random_state=42, verbosity=0)
    xgb.fit(X_train, y_train)
    models['XGBoost'] = ('raw', xgb)

# ============================================================
# EVALUATE ALL MODELS
# ============================================================
print(f'{"Model":25s} {"In-Sample":>12s} {"Out-of-Sample":>15s} {"Gap":>8s}')
print('=' * 65)

for name, (data_type, model) in models.items():
    if data_type == 'scaled':
        in_acc = accuracy_score(y_train, model.predict(X_train_scaled))
        out_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    else:
        in_acc = accuracy_score(y_train, model.predict(X_train))
        out_acc = accuracy_score(y_test, model.predict(X_test))
    
    gap = in_acc - out_acc
    results.append({'Model': name, 'In-Sample': in_acc, 'Out-of-Sample': out_acc, 'Gap': gap})
    print(f'{name:25s} {in_acc:11.1%} {out_acc:14.1%} {gap:+7.1%}')

results_df = pd.DataFrame(results)

print(f'\n{"Coin flip":25s} {"50.0%":>12s} {"50.0%":>15s}')
print('\nKey observation: Complex models have HIGHER in-sample accuracy')
print('but OUT-OF-SAMPLE, everything collapses toward 50%.')
print('The gap between in-sample and out-of-sample IS overfitting.')

In [ ]:
# ============================================================
# VISUALIZE: The Horse Race
# ============================================================
fig, ax = plt.subplots(figsize=(12, 7))

x_pos = np.arange(len(results_df))
width = 0.35

bars_in = ax.bar(x_pos - width/2, results_df['In-Sample'] * 100, width,
                 color='#1E2761', alpha=0.85, label='In-Sample (Training)', edgecolor='white')
bars_out = ax.bar(x_pos + width/2, results_df['Out-of-Sample'] * 100, width,
                  color='#F96167', alpha=0.85, label='Out-of-Sample (Test)', edgecolor='white')

# 50% line
ax.axhline(y=50, color='gray', linewidth=2, linestyle='--', alpha=0.7)
ax.text(len(results_df) - 0.5, 50.5, 'Coin Flip (50%)', fontsize=11, color='gray', ha='right')

# Value labels
for bar in bars_in:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, f'{h:.1f}%',
            ha='center', va='bottom', fontweight='bold', fontsize=11, color='#1E2761')
for bar in bars_out:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, f'{h:.1f}%',
            ha='center', va='bottom', fontweight='bold', fontsize=11, color='#F96167')

ax.set_xticks(x_pos)
ax.set_xticklabels(results_df['Model'], fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=14)
ax.set_title('Model Horse Race: Predicting SPY Direction (Up/Down)\n'
             'In-sample looks great... out-of-sample tells the truth',
             fontweight='bold', fontsize=14)
ax.legend(fontsize=12)
ax.set_ylim(45, max(results_df['In-Sample'].max() * 100, 75) + 5)

plt.tight_layout()
plt.show()

**Pause and reflect:**
- XGBoost might get 95%+ accuracy **in-sample** — it memorized the training data!
- But **out-of-sample**, it's barely better than a coin flip
- The gap between the two bars IS overfitting
- **This is the most important lesson in ML for finance**

---

## Part 5: Seeing Overfitting in Action

Let's make overfitting visible by varying model complexity (Random Forest depth) and plotting training vs. test accuracy.

In [ ]:
# Vary Random Forest depth and track accuracy
depths = [1, 2, 3, 5, 7, 10, 15, 20, None]  # None = unlimited
depth_labels = [str(d) if d else '∞' for d in depths]

train_accs = []
test_accs = []

for depth in depths:
    rf_d = RandomForestClassifier(n_estimators=200, max_depth=depth, random_state=42, n_jobs=-1)
    rf_d.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, rf_d.predict(X_train)))
    test_accs.append(accuracy_score(y_test, rf_d.predict(X_test)))

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(range(len(depths)), [a*100 for a in train_accs], 'o-', color='#1E2761', 
        linewidth=2.5, markersize=10, label='In-Sample (Training)', zorder=5)
ax.plot(range(len(depths)), [a*100 for a in test_accs], 's-', color='#F96167', 
        linewidth=2.5, markersize=10, label='Out-of-Sample (Test)', zorder=5)

# Overfitting shading
ax.fill_between(range(len(depths)),
                [a*100 for a in test_accs],
                [a*100 for a in train_accs],
                alpha=0.15, color='#F96167', label='Overfitting gap')

ax.axhline(y=50, color='gray', linewidth=2, linestyle='--', alpha=0.5)
ax.text(0.3, 50.5, 'Coin Flip', fontsize=10, color='gray')

ax.set_xticks(range(len(depths)))
ax.set_xticklabels(depth_labels, fontsize=12)
ax.set_xlabel('Random Forest Max Depth (Model Complexity →)', fontsize=14)
ax.set_ylabel('Accuracy (%)', fontsize=14)
ax.set_title('The Overfitting Trap\nTraining accuracy rises, but test accuracy stays flat',
             fontweight='bold', fontsize=14)
ax.legend(fontsize=12)
ax.set_ylim(45, 105)

plt.tight_layout()
plt.show()

best_idx = np.argmax(test_accs)
print(f'Best test accuracy: {test_accs[best_idx]:.1%} at max_depth={depths[best_idx]}')
print(f'Training accuracy at unlimited depth: {train_accs[-1]:.1%} (memorized!)')
print(f'\nThe gap between the lines IS overfitting.')
print('More complexity does NOT mean better predictions when signal is weak.')

---

## Part 6: Feature Importance — What Do the Models Look At?

Even though the models can't beat the market, it's instructive to see what features they think matter.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Lasso coefficients ---
ax = axes[0]
lasso_coefs = pd.Series(lasso.coef_[0], index=feature_cols)
lasso_sorted = lasso_coefs.reindex(lasso_coefs.abs().sort_values(ascending=True).index)
colors_l = ['#F96167' if c < 0 else '#028090' for c in lasso_sorted]
ax.barh(range(len(lasso_sorted)), lasso_sorted.values, color=colors_l, alpha=0.8)
ax.set_yticks(range(len(lasso_sorted)))
ax.set_yticklabels(lasso_sorted.index, fontsize=9)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Coefficient Value')
ax.set_title('Lasso (L1): Most Features Shrunk to ~0\n(Automatic feature selection)', fontweight='bold')
n_zero = (lasso_coefs.abs() < 0.01).sum()
ax.text(0.98, 0.02, f'{n_zero}/{len(lasso_coefs)} features\n≈ zero',
        transform=ax.transAxes, fontsize=12, ha='right', va='bottom',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='red'))

# --- Random Forest importance ---
ax = axes[1]
rf_imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
ax.barh(range(len(rf_imp)), rf_imp.values, color='#1E2761', alpha=0.8)
ax.set_yticks(range(len(rf_imp)))
ax.set_yticklabels(rf_imp.index, fontsize=9)
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Random Forest: Spreads Weight Across Features\n(May be fitting noise)', fontweight='bold')

fig.suptitle('What Do the Models Think Matters?', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Lasso is sparse — it keeps only a few features (good for noisy data).')
print('Random Forest uses everything — which can mean it\'s fitting noise.')

---

## Part 7: Does Prediction Accuracy = Profits?

Even a model with 52% accuracy sounds promising. But does it actually make money?

Let's simulate: if we trade based on each model's daily prediction (go long when it says "up", go short when it says "down"), how do cumulative returns compare to simply **buying and holding** SPY?

In [ ]:
# Get predictions for the test period
test_eval = test[['Log_Return']].copy()

# Logistic Regression signals
lr_pred = lr.predict(X_test_scaled)
test_eval['lr_signal'] = lr_pred * 2 - 1  # convert 0/1 to -1/+1

# Random Forest signals
rf_pred = rf.predict(X_test)
test_eval['rf_signal'] = rf_pred * 2 - 1

# Strategy returns: signal × actual return
# If we predicted UP (+1) and market went UP → positive return (correct!)
# If we predicted UP (+1) and market went DOWN → negative return (wrong)
test_eval['buy_hold'] = test_eval['Log_Return']
test_eval['lr_strategy'] = test_eval['lr_signal'] * test_eval['Log_Return']
test_eval['rf_strategy'] = test_eval['rf_signal'] * test_eval['Log_Return']

# Cumulative returns
fig, ax = plt.subplots(figsize=(14, 7))

cum_bh = test_eval['buy_hold'].cumsum() * 100
cum_lr = test_eval['lr_strategy'].cumsum() * 100
cum_rf = test_eval['rf_strategy'].cumsum() * 100

ax.plot(cum_bh.index, cum_bh.values, color='gray', linewidth=2.5, 
        label=f'Buy & Hold ({cum_bh.iloc[-1]:+.1f}%)')
ax.plot(cum_lr.index, cum_lr.values, color='#028090', linewidth=2, linestyle='--',
        label=f'Logistic Regression ({cum_lr.iloc[-1]:+.1f}%)')
ax.plot(cum_rf.index, cum_rf.values, color='#F96167', linewidth=2, linestyle='--',
        label=f'Random Forest ({cum_rf.iloc[-1]:+.1f}%)')

ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_xlabel('Date', fontsize=14)
ax.set_ylabel('Cumulative Log Return (%)', fontsize=14)
ax.set_title('Strategy Performance: ML Models vs. Buy & Hold (SPY)\n'
             'Out-of-sample — and this is BEFORE transaction costs!',
             fontweight='bold', fontsize=14)
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

print('Even if a model has slightly above 50% accuracy, it may not beat buy-and-hold.')
print('And we haven\'t even accounted for transaction costs (bid-ask spread, commissions).')
print('In practice, these costs would make the ML strategies even worse.')

---

## Part 8: Your Turn — Exercises

### Exercise 1: Try a Different Stock

Run the entire pipeline on a different stock or ETF. Try something with potentially different behavior:
- `TSLA` — very volatile
- `GLD` — gold ETF (commodity)
- `BTC-USD` — Bitcoin (crypto, potentially less efficient)
- `IWM` — small-cap stocks (less analyst coverage)

Does your stock show more or less predictability than SPY?

In [ ]:
# YOUR CODE HERE
# 1. Download your chosen ticker with yf.download(ticker, period='5y')
# 2. Flatten columns: df.columns = df.columns.get_level_values(0)
# 3. Calculate Log_Return
# 4. Run engineer_features() on your data
# 5. Split into train/test (last 252 days = test)
# 6. Train at least 2 models and compare in-sample vs. out-of-sample
# 7. Plot the horse race chart



### Exercise 2: Does More Data Help?

We used 5 years of data. What if you use 10 years? Or just 2 years? Does the amount of training data change the out-of-sample results?

Try downloading `SPY` with `period='10y'` and `period='2y'`, and compare the out-of-sample accuracy.

In [ ]:
# YOUR CODE HERE
# Hint: You can reuse engineer_features() on the new data
# Compare the out-of-sample accuracy across different training set sizes



### Exercise 3: Add Your Own Feature

Can you think of a feature that might help? Some ideas:
- **Volume ratio**: Today's volume / 20-day average volume (unusual activity?)
- **Price range**: (High - Low) / Close (how much intraday movement?)
- **Gap**: Open vs. previous Close (overnight moves)
- **Day of week**: Is Monday different from Friday?

Add a feature to `engineer_features()`, retrain the models, and see if out-of-sample accuracy improves.

In [ ]:
# YOUR CODE HERE
# Hint: Modify the engineer_features function to add your new feature
# Don't forget to lag it! (We can only use past data, not future data)



### Exercise 4: Predicting Volatility Instead of Direction

We saw that volatility is more predictable than direction. Try predicting whether **tomorrow's volatility will be above or below average** instead of predicting direction.

Steps:
1. Create a new target: `high_vol = 1` if today's |return| > median |return|, else 0
2. Use the same features
3. Train a model and compare in-sample vs. out-of-sample accuracy

Is volatility genuinely more predictable?

In [ ]:
# YOUR CODE HERE
# Hint: 
# median_abs_ret = df_ml['Log_Return'].abs().median()
# df_ml['high_vol'] = (df_ml['Log_Return'].abs() > median_abs_ret).astype(int)
# Then retrain your models with 'high_vol' as the target



### Exercise 5: The Data Snooping Trap

Here's a scary experiment: what if you test **hundreds of random "strategies"** on the same data? Some will look good *by pure chance*.

Generate 200 random trading signals (coin flips), compute the accuracy of each against the actual test labels, and plot the distribution. How many achieve >52% accuracy? >54%?

This is the **multiple testing problem**: if you try enough strategies, you *will* find something that looks like it works — even on random noise.

In [ ]:
# YOUR CODE HERE
# Hint:
# np.random.seed(42)
# n_strategies = 200
# n_test = len(y_test)
# random_accs = []
# for _ in range(n_strategies):
#     random_preds = np.random.randint(0, 2, size=n_test)
#     acc = accuracy_score(y_test, random_preds)
#     random_accs.append(acc)
#
# Then plot a histogram of random_accs
# Add vertical lines for 50% (coin flip) and for your best model's out-of-sample accuracy
# How many random strategies beat your actual model?


### Exercise 6: Walk-Forward Validation

A single train/test split can be misleading — what if the test year happened to be "easy"? **Walk-forward validation** uses a rolling window: train on the past N days, predict the next day, then slide the window forward.

Implement walk-forward validation with a 500-day training window and a Logistic Regression model. Plot the rolling 60-day accuracy over time.

Does accuracy stay stable, or does it vary with market conditions (regimes)?

In [ ]:
# YOUR CODE HERE
# Hint:
# train_window = 500
# predictions = []
# actuals = []
# dates = []
#
# for i in range(train_window, len(df_ml)):
#     X_wf_train = df_ml[feature_cols].iloc[i-train_window:i]
#     y_wf_train = df_ml['target'].iloc[i-train_window:i]
#     X_wf_test = df_ml[feature_cols].iloc[[i]]
#     y_wf_test = df_ml['target'].iloc[i]
#
#     scaler_wf = StandardScaler()
#     X_wf_train_s = scaler_wf.fit_transform(X_wf_train)
#     X_wf_test_s = scaler_wf.transform(X_wf_test)
#
#     lr_wf = LogisticRegression(max_iter=1000, random_state=42)
#     lr_wf.fit(X_wf_train_s, y_wf_train)
#     pred = lr_wf.predict(X_wf_test_s)[0]
#
#     predictions.append(pred)
#     actuals.append(y_wf_test)
#     dates.append(df_ml.index[i])
#
# Then compute rolling 60-day accuracy:
# rolling_acc = pd.Series((np.array(predictions) == np.array(actuals)).astype(float),
#                         index=dates).rolling(60).mean()
# Plot rolling_acc with a horizontal line at 0.5


### Exercise 7: Confusion Matrix — What Does ~52% Actually Look Like?

Build a **confusion matrix** for your best model's out-of-sample predictions. Use `sklearn.metrics.confusion_matrix` and `ConfusionMatrixDisplay`.

Then compare it to a confusion matrix from pure random guessing (coin flip). Can you visually tell the difference?

This connects to the lecture: even a 2% edge above coin-flip is valuable in finance — but only with proper position sizing (the Kelly Criterion) and very low transaction costs.

In [ ]:
# YOUR CODE HERE
# Hint:
# from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
#
# # Get predictions from your best model (e.g., Random Forest)
# rf_preds = rf.predict(X_test)
#
# # Random baseline
# np.random.seed(0)
# random_preds = np.random.randint(0, 2, size=len(y_test))
#
# fig, axes = plt.subplots(1, 2, figsize=(12, 5))
#
# # Left: Your model
# cm_model = confusion_matrix(y_test, rf_preds)
# ConfusionMatrixDisplay(cm_model, display_labels=['Down', 'Up']).plot(ax=axes[0], cmap='Blues')
# axes[0].set_title(f'Random Forest ({accuracy_score(y_test, rf_preds):.1%})')
#
# # Right: Coin flip
# cm_random = confusion_matrix(y_test, random_preds)
# ConfusionMatrixDisplay(cm_random, display_labels=['Down', 'Up']).plot(ax=axes[1], cmap='Reds')
# axes[1].set_title(f'Coin Flip ({accuracy_score(y_test, random_preds):.1%})')
#
# plt.tight_layout()
# plt.show()


### Exercise 8: Prompt Engineering — Connect Your Results to the Lecture

Write a prompt for an AI assistant (Claude, ChatGPT, etc.) that asks it to connect your lab results to the key lecture concepts. Your prompt should include your actual numbers from the lab and ask the AI to address:

- Do your results support **weak-form market efficiency**? Why or why not?
- What would it look like if markets were **NOT** efficient? (What accuracy and R² would you expect?)
- How do your results relate to the **Grossman-Stiglitz paradox** — if no one can profit from information, why would anyone bother gathering it?
- If Renaissance Technologies achieves a ~66% annual return, what must they be doing differently than what we tried here?

**Tips for a good prompt:** Include specific numbers (your R², out-of-sample accuracy, overfitting gap). Tell the AI what audience you're writing for (e.g., "explain to an MBA student"). Ask it to be concrete, not generic.

---

## Key Takeaways from This Lab

1. **Returns show no autocorrelation** — past returns don't predict future direction (supports weak-form EMH)
2. **Volatility IS predictable** — squared returns show strong autocorrelation (this is exploitable!)
3. **Feature engineering** transforms raw prices into richer inputs (SMA, momentum, volatility)
4. **More complex models overfit more** — XGBoost memorizes training data but fails on new data
5. **In-sample performance is meaningless** — always evaluate out-of-sample
6. **~50% accuracy for direction prediction** — even with sophisticated ML, it's nearly a coin flip
7. **Beating buy-and-hold is extremely hard** — especially after transaction costs
8. **Data snooping is dangerous** — test enough random strategies and some will look significant by pure chance (multiple testing problem)
9. **Walk-forward validation** is essential — a single train/test split can hide how accuracy varies across market regimes
10. **The smart money** focuses on: better data (not better models), predicting volatility (not direction), proper position sizing (Kelly Criterion), and rigorous walk-forward validation

```
YOUR PROMPT HERE:



```